# 04 · Product Profitability

**Goal:** see which products and categories earn the most revenue and the most estimated profit.

**Source:** `gold.report_products` (130 products)

**Important:** profit is an *estimate*. Cost of goods = product cost × quantity sold. Discounts, freight and overheads are not included.

**Output:** product profitability table, category summary and a Pareto chart in `output/`

## 1. Setup

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import config

sns.set_style("whitegrid")
os.makedirs("output", exist_ok=True)

## 2. Load products

In [4]:
engine = create_engine(config.get_conn_url())
prod = pd.read_sql("SELECT * FROM gold.report_products", engine)
print("Rows loaded:", len(prod))
print(
    prod[["product_name", "category", "cost", "total_sales", "total_quantity"]].head()
)

Rows loaded: 130
             product_name     category  cost  total_sales  total_quantity
0  All-Purpose Bike Stand  Accessories    59        39591             249
1            AWC Logo Cap     Clothing     7        19710            2190
2   Bike Wash - Dissolver  Accessories     3         7272             909
3         Classic Vest- L     Clothing    24        12480             195
4         Classic Vest- M     Clothing    24        12736             199


## 3. Estimated cost, profit and margin

- `est_cost` = cost × total quantity
- `est_profit` = total sales − `est_cost`
- `est_margin_pct` = `est_profit` ÷ total sales × 100

In [5]:
prod["est_cost"] = prod["cost"] * prod["total_quantity"]
prod["est_profit"] = prod["total_sales"] - prod["est_cost"]
prod["est_margin_pct"] = (
    100 * prod["est_profit"] / prod["total_sales"].replace(0, np.nan)
)
print(prod["est_margin_pct"].describe())

count    130.000000
mean      44.035806
std       12.296800
min       22.222222
25%       36.296296
50%       39.323644
75%       48.871681
max       75.000000
Name: est_margin_pct, dtype: float64


Estimated margin per product ranges from 22.2% to 75.0% (median 39.3%).

## 4. Category economics

In [6]:
cat_summary = (
    prod.groupby("category")
    .agg(
        total_sales=("total_sales", "sum"),
        total_est_profit=("est_profit", "sum"),
        products=("product_name", "nunique"),
        units_sold=("total_quantity", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)
cat_summary["est_margin_pct"] = (
    100
    * cat_summary["total_est_profit"]
    / cat_summary["total_sales"].replace(0, np.nan)
).round(1)
cat_summary["revenue_share_%"] = (
    100 * cat_summary["total_sales"] / cat_summary["total_sales"].sum()
).round(1)
print(cat_summary)

             total_sales  total_est_profit  products  units_sold  \
category                                                           
Bikes           28316272          11109565        88       15205   
Accessories       700262            439510        22       36112   
Clothing          339716            136682        20        9106   

             est_margin_pct  revenue_share_%  
category                                      
Bikes                  39.2             96.5  
Accessories            62.8              2.4  
Clothing               40.2              1.2  


Bikes bring 96.5% of revenue at a 39.2% estimated margin. Accessories have the highest margin (62.8%) but only 2.4% of revenue. Clothing is at 40.2% margin and 1.2% of revenue.

## 5. Pareto analysis (80% of revenue)

Products are ranked by revenue and the cumulative revenue share is tracked, to find how many products make up 80% of sales.

In [7]:
p = prod.sort_values("total_sales", ascending=False).reset_index(drop=True)
p["cum_revenue"] = p["total_sales"].cumsum()
p["cum_revenue_pct"] = 100 * p["cum_revenue"] / p["total_sales"].sum()
p["product_rank_pct"] = 100 * (p.index + 1) / len(p)
crossed_80 = p.index[p["cum_revenue_pct"] >= 80]
n80_index = int(crossed_80[0]) if len(crossed_80) else len(p) - 1
n80 = n80_index + 1
share_products = round(100 * n80 / len(p), 1)
print(
    f"{n80} products ({share_products}% of catalogue) reach at least 80% cumulative revenue."
)

35 products (26.9% of catalogue) reach at least 80% cumulative revenue.


35 products (26.9% of the catalogue) reach at least 80% of cumulative revenue.

## 6. Portfolio quadrants: revenue vs. estimated margin

Products are split at the median revenue and the median margin.

In [9]:
revenue_median = prod["total_sales"].median()
margin_median = prod["est_margin_pct"].median()


def quadrant(row):
    high_rev = row["total_sales"] >= revenue_median
    high_margin = row["est_margin_pct"] >= margin_median
    if high_rev and high_margin:
        return "High Revenue / High Margin"
    if high_rev and not high_margin:
        return "High Revenue / Lower Margin"
    if not high_rev and high_margin:
        return "Lower Revenue / High Margin"
    return "Lower Revenue / Lower Margin"


prod["portfolio_quadrant"] = prod.apply(quadrant, axis=1)
print(prod["portfolio_quadrant"].value_counts())

portfolio_quadrant
Lower Revenue / High Margin     40
High Revenue / Lower Margin     39
High Revenue / High Margin      26
Lower Revenue / Lower Margin    25
Name: count, dtype: int64


Quadrant counts: Lower Revenue / High Margin 40 · High Revenue / Lower Margin 39 · High Revenue / High Margin 26 · Lower Revenue / Lower Margin 25.

## 7. Pareto chart

In [11]:
plt.figure(figsize=(9, 5))
plt.plot(p["product_rank_pct"], p["cum_revenue_pct"])
plt.axhline(80, linestyle="--", label="80% Revenue")
plt.axvline(
    p.loc[n80_index, "product_rank_pct"], linestyle="--", label=f"{n80} Products"
)
plt.title("Pareto Curve — Product Contribution to Revenue")
plt.xlabel("% of Products (ranked by revenue)")
plt.ylabel("Cumulative % of Revenue")
plt.legend()
plt.tight_layout()
plt.savefig("output/07_pareto_curve_v2.png", dpi=120)
plt.close()

![Pareto curve of product revenue](output/07_pareto_curve_v2.png)

## 8. Save outputs

In [12]:
prod.to_csv("output/product_profitability_v2.csv", index=False)
cat_summary.to_csv("output/category_margin_summary_v2.csv")
print("Product profitability analysis complete.")

Product profitability analysis complete.
